In [1]:
import pennylane as qp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("bank_marketing_qos_encoded.csv")

In [2]:
rng = np.random.default_rng(42)

yes_indices = df.index[df["y"] == "yes"].to_numpy()
no_indices = df.index[df["y"] == "no"].to_numpy()

rng.shuffle(yes_indices)
rng.shuffle(no_indices)

yes_split = int(0.8 * len(yes_indices))
no_split = int(0.8 * len(no_indices))

train_indices = np.concatenate([
    yes_indices[:yes_split],
    no_indices[:no_split]
])

test_indices = np.concatenate([
    yes_indices[yes_split:],
    no_indices[no_split:]
])

train_df = df.loc[train_indices]

test_df = (
    df.loc[test_indices]
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 36168
Test: 9043


In [3]:
num_addresses = 240
quantum_dimension = 256
num_qubits = 8

alpha = 1.0

train_yes = train_df[train_df["y"] == "yes"]
train_no = train_df[train_df["y"] == "no"]

counts_yes = np.bincount(
    train_yes["qos_address"].to_numpy(),
    minlength=num_addresses
)

counts_no = np.bincount(
    train_no["qos_address"].to_numpy(),
    minlength=num_addresses
)

p_yes_train = (
    counts_yes + alpha
) / (
    counts_yes.sum()
    + alpha * num_addresses
)

p_no_train = (
    counts_no + alpha
) / (
    counts_no.sum()
    + alpha * num_addresses
)

p_yes_padded = np.zeros(quantum_dimension)
p_no_padded = np.zeros(quantum_dimension)

p_yes_padded[:num_addresses] = p_yes_train
p_no_padded[:num_addresses] = p_no_train

In [5]:
def qos_phase_state(probabilities, t):
    return (
        np.exp(1j * t * probabilities)
        / np.sqrt(len(probabilities))
    )

In [6]:
t_kernel = 20.0

yes_class_state = qos_phase_state(
    p_yes_padded,
    t_kernel
)

no_class_state = qos_phase_state(
    p_no_padded,
    t_kernel
)

In [7]:
def customer_qos_state(
    address,
    t,
    dimension=256
):
    probabilities = np.zeros(dimension)

    probabilities[int(address)] = 1.0

    return qos_phase_state(
        probabilities,
        t
    )

In [8]:
example_address = int(
    test_df["qos_address"].iloc[0]
)

example_customer_state = customer_qos_state(
    example_address,
    t_kernel
)

print("Address:", example_address)
print(
    "Norm:",
    np.sum(np.abs(example_customer_state)**2)
)

Address: 39
Norm: 1.0


In [9]:
def state_kernel(state_a, state_b):
    overlap = np.vdot(
        state_a,
        state_b
    )

    return float(
        np.abs(overlap) ** 2
    )

In [10]:
score_yes = state_kernel(
    example_customer_state,
    yes_class_state
)

score_no = state_kernel(
    example_customer_state,
    no_class_state
)

print("Example address:", example_address)
print("Yes kernel score:", score_yes)
print("No kernel score:", score_no)

Example address: 39
Yes kernel score: 0.9695489186525543
No kernel score: 0.9728622603576146


In [11]:
wires = list(range(num_qubits))

dev = qp.device(
    "default.qubit",
    wires=num_qubits
)

In [12]:
@qp.qnode(dev)
def customer_kernel_state(
    probabilities,
    t
):
    for wire in wires:
        qp.Hadamard(wires=wire)

    diagonal = np.exp(
        1j * t * probabilities
    )

    qp.DiagonalQubitUnitary(
        diagonal,
        wires=wires
    )

    return qp.state()

In [13]:
customer_distribution = np.zeros(
    quantum_dimension
)

customer_distribution[
    example_address
] = 1.0

In [14]:
customer_state_qnode = customer_kernel_state(
    customer_distribution,
    t_kernel
)

print(
    "Analytic vs PennyLane fidelity:",
    np.abs(
        np.vdot(
            example_customer_state,
            customer_state_qnode
        )
    ) ** 2
)

Analytic vs PennyLane fidelity: 0.9999999999999996


In [15]:
def predict_kernel_customer(
    address,
    yes_state,
    no_state,
    t
):

    customer_state = customer_qos_state(
        address,
        t
    )

    yes_score = state_kernel(
        customer_state,
        yes_state
    )

    no_score = state_kernel(
        customer_state,
        no_state
    )

    prediction = (
        "yes"
        if yes_score > no_score
        else "no"
    )

    return prediction, yes_score, no_score

In [16]:
for i in range(10):

    row = test_df.iloc[i]

    address = int(
        row["qos_address"]
    )

    true_label = row["y"]

    pred, yes_score, no_score = (
        predict_kernel_customer(
            address,
            yes_class_state,
            no_class_state,
            t_kernel
        )
    )

    print(
        f"Address={address:3d} | "
        f"True={true_label:3s} | "
        f"Yes={yes_score:.8f} | "
        f"No={no_score:.8f} | "
        f"Pred={pred}"
    )

Address= 39 | True=no  | Yes=0.96954892 | No=0.97286226 | Pred=no
Address= 52 | True=yes | Yes=0.97138165 | No=0.97471450 | Pred=no
Address=144 | True=no  | Yes=0.97117380 | No=0.97354909 | Pred=no
Address=126 | True=no  | Yes=0.97053217 | No=0.97449053 | Pred=no
Address=112 | True=no  | Yes=0.97441987 | No=0.97588190 | Pred=no
Address=134 | True=no  | Yes=0.97100243 | No=0.97537489 | Pred=no
Address=182 | True=no  | Yes=0.97007750 | No=0.97404108 | Pred=no
Address=204 | True=yes | Yes=0.97001392 | No=0.97280232 | Pred=no
Address= 59 | True=no  | Yes=0.96960967 | No=0.97279805 | Pred=no
Address= 54 | True=no  | Yes=0.97396981 | No=0.98202995 | Pred=no


In [17]:
uniform_state = (
    np.ones(quantum_dimension, dtype=complex)
    / np.sqrt(quantum_dimension)
)

print(
    "Uniform-state norm:",
    np.sum(np.abs(uniform_state) ** 2)
)

Uniform-state norm: 1.0


In [18]:
def center_state(state, uniform_state):
    """
    Remove the component of `state` parallel to the uniform state.
    """

    projection_amplitude = np.vdot(
        uniform_state,
        state
    )

    centered = (
        state
        - projection_amplitude * uniform_state
    )

    norm = np.linalg.norm(centered)

    if norm < 1e-12:
        raise ValueError(
            "Centered state has near-zero norm."
        )

    return centered / norm

In [19]:
yes_class_state_centered = center_state(
    yes_class_state,
    uniform_state
)

no_class_state_centered = center_state(
    no_class_state,
    uniform_state
)

print(
    "Centered yes norm:",
    np.linalg.norm(yes_class_state_centered)
)

print(
    "Centered no norm:",
    np.linalg.norm(no_class_state_centered)
)

Centered yes norm: 1.0000000000000002
Centered no norm: 0.9999999999999999


In [20]:
example_customer_centered = center_state(
    example_customer_state,
    uniform_state
)

print(
    "Centered customer norm:",
    np.linalg.norm(example_customer_centered)
)

Centered customer norm: 0.9999999999999997


In [21]:
centered_yes_score = state_kernel(
    example_customer_centered,
    yes_class_state_centered
)

centered_no_score = state_kernel(
    example_customer_centered,
    no_class_state_centered
)

print(
    "Centered yes kernel score:",
    centered_yes_score
)

print(
    "Centered no kernel score:",
    centered_no_score
)

Centered yes kernel score: 0.0005199069734744434
Centered no kernel score: 0.00018354069664043405


In [22]:
def predict_centered_kernel_customer(
    address,
    t,
    yes_class_state_centered,
    no_class_state_centered,
    uniform_state
):

    customer_state = customer_qos_state(
        address,
        t
    )

    customer_centered = center_state(
        customer_state,
        uniform_state
    )

    yes_score = state_kernel(
        customer_centered,
        yes_class_state_centered
    )

    no_score = state_kernel(
        customer_centered,
        no_class_state_centered
    )

    prediction = (
        "yes"
        if yes_score > no_score
        else "no"
    )

    return prediction, yes_score, no_score

In [23]:
for i in range(10):

    row = test_df.iloc[i]

    address = int(
        row["qos_address"]
    )

    true_label = row["y"]

    pred, yes_score, no_score = (
        predict_centered_kernel_customer(
            address=address,
            t=t_kernel,
            yes_class_state_centered=yes_class_state_centered,
            no_class_state_centered=no_class_state_centered,
            uniform_state=uniform_state
        )
    )

    print(
        f"Address={address:3d} | "
        f"True={true_label:3s} | "
        f"Yes={yes_score:.6f} | "
        f"No={no_score:.6f} | "
        f"Pred={pred}"
    )

Address= 39 | True=no  | Yes=0.000520 | No=0.000184 | Pred=yes
Address= 52 | True=yes | Yes=0.005684 | No=0.008282 | Pred=no
Address=144 | True=no  | Yes=0.004226 | No=0.000773 | Pred=yes
Address=126 | True=no  | Yes=0.001035 | No=0.006270 | Pred=no
Address=112 | True=no  | Yes=0.047144 | No=0.022681 | Pred=yes
Address=134 | True=no  | Yes=0.003175 | No=0.015649 | Pred=no
Address=182 | True=no  | Yes=0.000080 | No=0.003025 | Pred=no
Address=204 | True=yes | Yes=0.000041 | No=0.000289 | Pred=no
Address= 59 | True=no  | Yes=0.000377 | No=0.000297 | Pred=yes
Address= 54 | True=no  | Yes=0.038798 | No=0.196812 | Pred=no


In [24]:
centered_predictions = []

for address in test_df["qos_address"].to_numpy():

    pred, _, _ = (
        predict_centered_kernel_customer(
            address=int(address),
            t=t_kernel,
            yes_class_state_centered=yes_class_state_centered,
            no_class_state_centered=no_class_state_centered,
            uniform_state=uniform_state
        )
    )

    centered_predictions.append(pred)

centered_predictions = np.array(
    centered_predictions
)

In [25]:
true_labels = test_df["y"].to_numpy()

yes_mask = true_labels == "yes"
no_mask = true_labels == "no"

yes_accuracy_centered = np.mean(
    centered_predictions[yes_mask] == "yes"
)

no_accuracy_centered = np.mean(
    centered_predictions[no_mask] == "no"
)

balanced_accuracy_centered = (
    yes_accuracy_centered
    + no_accuracy_centered
) / 2

overall_accuracy_centered = np.mean(
    centered_predictions == true_labels
)

print(
    "Yes-class accuracy:",
    yes_accuracy_centered
)

print(
    "No-class accuracy:",
    no_accuracy_centered
)

print(
    "Balanced accuracy:",
    balanced_accuracy_centered
)

print(
    "Overall accuracy:",
    overall_accuracy_centered
)

Yes-class accuracy: 0.5614366729678639
No-class accuracy: 0.6345648090169067
Balanced accuracy: 0.5980007409923853
Overall accuracy: 0.6260090677872387
